# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliAlmasri7/FlyRank-ML/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions and reason codes

### Rule: Prioritize pages for human review

I rank content pages using a combination of observed search performance, search visibility, and a model-based signal. The goal is to help a reviewer decide which pages deserve attention first, not to automatically prescribe a content change.

The analysis uses March 2026 as the feature window and April 2026 as the future target window. Each row represents one pseudonymized client-content pair.

The queue considers four signals:

1. **CTR below position peers:** the page’s observed CTR is below the average CTR of pages in the same average-position group.
2. **Striking-distance opportunity:** the page’s average position is between 11 and 20, making it a candidate for further review.
3. **High visibility:** the page has substantial impressions, so a review may be more useful than reviewing a page with very little exposure.
4. **Model signal:** the predicted future CTR differs materially from the observed CTR, flagging a page for investigation rather than asserting that its performance will improve or decline.

Pages with at least 100 March impressions are eligible for the main review queue. The priority score combines the size of the positive CTR gap, search visibility, and any additional review signals. Reason codes explain the main reason for each recommendation.

These thresholds are operational choices for this exercise, not universal SEO rules. Low CTR does not prove that a title, meta description, or content is the cause. All recommendations require human review, and the observed associations do not establish causation.

In [2]:
import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from dotenv import load_dotenv
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit

# ------------------------------------------------------------
# 1. Authenticate with Hugging Face
# ------------------------------------------------------------

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. Set it as an environment variable "
        "or in your .env file, and make sure you have accepted "
        "the FlyRank dataset access terms."
    )

con = duckdb.connect()

# Escape quotes without displaying the token.
safe_token = HF_TOKEN.replace("'", "''")

con.execute(
    f"CREATE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{safe_token}')"
)

# ------------------------------------------------------------
# 2. Define the March feature window and April target window
# ------------------------------------------------------------

rel = "hf://datasets/FlyRank/internship-warehouse"

march_path = (
    f"{rel}/fact_content_daily_performance/"
    "month=2026-03/*.parquet"
)

april_path = (
    f"{rel}/fact_content_daily_performance/"
    "month=2026-04/*.parquet"
)

# ------------------------------------------------------------
# 3. Aggregate March to one row per client-content pair
# ------------------------------------------------------------

march_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        SUM(gsc_sum_position) AS gsc_sum_position,

        SUM(gsc_sum_position) * 1.0
            / NULLIF(SUM(gsc_impressions), 0)
            AS gsc_avg_position

    FROM read_parquet('{march_path}')

    WHERE gsc_impressions > 0

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING
        SUM(gsc_impressions) > 0
        AND SUM(gsc_sum_position) IS NOT NULL
""").df()

march_df["observed_ctr"] = (
    march_df["gsc_clicks"] / march_df["gsc_impressions"]
)

march_df = march_df.replace(
    [np.inf, -np.inf], np.nan
).dropna(
    subset=["gsc_avg_position", "observed_ctr"]
)

# ------------------------------------------------------------
# 4. Aggregate April separately to create the future target
# ------------------------------------------------------------

april_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS future_impressions,
        SUM(gsc_clicks) AS future_clicks

    FROM read_parquet('{april_path}')

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

april_df["future_ctr"] = (
    april_df["future_clicks"]
    / april_df["future_impressions"].replace(0, np.nan)
)

print("March page rows:", len(march_df))
print("April target rows:", len(april_df))

# ------------------------------------------------------------
# 5. Define position groups
# ------------------------------------------------------------

def get_position_group(position):
    if position <= 3:
        return "1-3"
    elif position <= 10:
        return "4-10"
    elif position <= 20:
        return "11-20"
    elif position <= 50:
        return "21-50"
    return "50+"

march_df["position_group"] = (
    march_df["gsc_avg_position"].apply(get_position_group)
)

# ------------------------------------------------------------
# 6. Build the labeled dataset for training and validation
# ------------------------------------------------------------

labeled_df = march_df.merge(
    april_df,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Keep the same minimum future-impression rule used in
# the earlier model evaluation.
labeled_df = labeled_df[
    (labeled_df["future_impressions"] >= 10)
    & labeled_df["future_ctr"].notna()
].copy()

print("Labeled rows:", len(labeled_df))

# April fields are targets/validation context, not model features.
feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "observed_ctr",
    "gsc_avg_position",
    "position_group"
]

X_all = pd.get_dummies(
    labeled_df[feature_columns],
    columns=["position_group"],
    dtype=int
)

feature_names = X_all.columns.tolist()
y = labeled_df["future_ctr"].astype(float)

# ------------------------------------------------------------
# 7. Split by client to reduce client-level leakage
# ------------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, valid_idx = next(
    splitter.split(
        X_all,
        y,
        groups=labeled_df["client_hash_id"]
    )
)

X_train = X_all.iloc[train_idx]
X_valid = X_all.iloc[valid_idx]

y_train = y.iloc[train_idx]
y_valid = y.iloc[valid_idx]

heldout_clients = set(
    labeled_df.iloc[valid_idx]["client_hash_id"]
)

print("Training rows:", len(train_idx))
print("Validation rows:", len(valid_idx))
print("Held-out clients:", len(heldout_clients))

# ------------------------------------------------------------
# 8. Train the model
# ------------------------------------------------------------

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

valid_predictions = np.clip(
    model.predict(X_valid), 0, 1
)

rho = spearmanr(y_valid, valid_predictions).statistic

print(f"Validation Spearman correlation: {rho:.4f}")

# ------------------------------------------------------------
# 9. Build a review queue from March data for held-out clients
# ------------------------------------------------------------

# Important: use March features to select and score queue items.
# Do not filter queue items using April impressions or April CTR.
queue_candidates = march_df[
    march_df["client_hash_id"].isin(heldout_clients)
].copy()

# Create the same one-hot feature columns used for training.
X_queue = pd.get_dummies(
    queue_candidates[feature_columns],
    columns=["position_group"],
    dtype=int
)

X_queue = X_queue.reindex(
    columns=feature_names,
    fill_value=0
)

queue_candidates["ml_prediction"] = np.clip(
    model.predict(X_queue), 0, 1
)

# ------------------------------------------------------------
# 10. Calculate position-peer benchmarks using March only
# ------------------------------------------------------------

position_benchmark = (
    march_df.groupby("position_group")["observed_ctr"]
    .median()
    .rename("peer_median_ctr")
)

queue_candidates = queue_candidates.join(
    position_benchmark,
    on="position_group"
)

queue_candidates["ctr_gap"] = (
    queue_candidates["peer_median_ctr"]
    - queue_candidates["observed_ctr"]
)

# This compares the estimated future CTR with the current CTR.
# It is a review signal, NOT an estimate of causal improvement.
queue_candidates["predicted_ctr_gap"] = (
    queue_candidates["ml_prediction"]
    - queue_candidates["observed_ctr"]
)

queue_candidates["abs_predicted_ctr_gap"] = (
    queue_candidates["predicted_ctr_gap"].abs()
)

# ------------------------------------------------------------
# 11. Define eligibility and human-readable reason codes
# ------------------------------------------------------------

MIN_IMPRESSIONS = 100
HIGH_VISIBILITY_IMPRESSIONS = 1000
MODEL_GAP_THRESHOLD = 0.005

queue_candidates["eligible_for_review"] = (
    queue_candidates["gsc_impressions"] >= MIN_IMPRESSIONS
)

# Assign one primary reason code using a fixed priority order.
conditions = [
    (
        queue_candidates["eligible_for_review"]
        & (queue_candidates["ctr_gap"] > 0)
    ),
    (
        queue_candidates["eligible_for_review"]
        & queue_candidates["gsc_avg_position"].between(11, 20)
    ),
    (
        queue_candidates["eligible_for_review"]
        & (
            queue_candidates["gsc_impressions"]
            >= HIGH_VISIBILITY_IMPRESSIONS
        )
    ),
    (
        queue_candidates["eligible_for_review"]
        & (
            queue_candidates["abs_predicted_ctr_gap"]
            >= MODEL_GAP_THRESHOLD
        )
    )
]

reason_values = [
    "CTR_BELOW_POSITION_PEERS",
    "STRIKING_DISTANCE_REVIEW",
    "HIGH_VISIBILITY_REVIEW",
    "MODEL_SIGNAL_REVIEW"
]

queue_candidates["reason_code"] = np.select(
    conditions,
    reason_values,
    default="MONITOR_NO_CLEAR_FLAG"
)

action_map = {
    "CTR_BELOW_POSITION_PEERS": "REVIEW_SNIPPET",
    "STRIKING_DISTANCE_REVIEW": "REVIEW_CONTENT",
    "HIGH_VISIBILITY_REVIEW": "REVIEW_PERFORMANCE",
    "MODEL_SIGNAL_REVIEW": "REVIEW_TREND",
    "MONITOR_NO_CLEAR_FLAG": "MONITOR"
}

queue_candidates["recommended_action"] = (
    queue_candidates["reason_code"].map(action_map)
)

# ------------------------------------------------------------
# 12. Create a transparent priority score
# ------------------------------------------------------------

# A positive peer CTR gap contributes to the score.
# A model discrepancy adds a smaller investigation component.
# log1p(impressions) gives visibility more weight without allowing
# very large impression counts to dominate linearly.

visibility_weight = np.log1p(
    queue_candidates["gsc_impressions"]
)

queue_candidates["priority_score"] = (
    queue_candidates["ctr_gap"].clip(lower=0)
    * visibility_weight
    + 0.25
    * queue_candidates["abs_predicted_ctr_gap"]
    * visibility_weight
)

# Keep only pages that meet the minimum evidence threshold.
queue = queue_candidates[
    queue_candidates["eligible_for_review"]
].copy()

queue = queue.sort_values(
    ["priority_score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

queue.insert(0, "rank", np.arange(1, len(queue) + 1))

# ------------------------------------------------------------
# 13. Add confidence notes for reviewers
# ------------------------------------------------------------

queue["evidence_note"] = np.select(
    [
        queue["gsc_impressions"] >= 1000,
        queue["gsc_impressions"] >= 300
    ],
    [
        "Higher observed visibility; still requires review",
        "Moderate evidence; check context before acting"
    ],
    default="Limited evidence; interpret cautiously"
)

# ------------------------------------------------------------
# 14. Save the queue
# ------------------------------------------------------------

# Works whether the notebook is run from the repository root
# or from work/notebooks/.
cwd = Path.cwd()

possible_roots = [cwd, *cwd.parents]
repo_root = next(
    (
        path for path in possible_roots
        if (path / "work" / "notebooks").exists()
    ),
    cwd
)

output_dir = repo_root / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "w07_ranked_action_queue.csv"

output_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "observed_ctr",
    "gsc_avg_position",
    "position_group",
    "peer_median_ctr",
    "ctr_gap",
    "ml_prediction",
    "predicted_ctr_gap",
    "priority_score",
    "reason_code",
    "recommended_action",
    "evidence_note"
]

queue[output_columns].to_csv(output_path, index=False)

print("Saved queue to:", output_path)
print("Total ranked pages:", len(queue))

display(queue[output_columns].head(20))

March page rows: 176738
April target rows: 362172
Labeled rows: 137742
Training rows: 107945
Validation rows: 29797
Held-out clients: 9
Validation Spearman correlation: 0.3761
Saved queue to: c:\Users\Maxs_Z\Documents\FlyRank-ML\work\outputs\w07_ranked_action_queue.csv
Total ranked pages: 23496


,rank,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,observed_ctr,gsc_avg_position,position_group,peer_median_ctr,ctr_gap,ml_prediction,predicted_ctr_gap,priority_score,reason_code,recommended_action,evidence_note
0,1,client_f623b01661d4bfe4,content_0a124186f945b8cd,259.0,19.0,0.073359,18.444015,11-20,0.0,-0.073359,0.031635,-0.041724,0.058004,STRIKING_DISTANCE_REVIEW,REVIEW_CONTENT,Limited evidence; interpret cautiously
1,2,client_f623b01661d4bfe4,content_b96495c6b2911b12,160.0,12.0,0.075000,33.693750,21-50,0.0,-0.075000,0.031599,-0.043401,0.055134,MODEL_SIGNAL_REVIEW,REVIEW_TREND,Limited evidence; interpret cautiously
2,3,client_2094c6eb080311d5,content_04e3473c74438c83,140.0,11.0,0.078571,2.850000,1-3,0.0,-0.078571,0.035175,-0.043397,0.053690,MODEL_SIGNAL_REVIEW,REVIEW_TREND,Limited evidence; interpret cautiously
3,4,client_f623b01661d4bfe4,content_7b50820e76b1a006,2243.0,103.0,0.045921,29.108337,21-50,0.0,-0.045921,0.024751,-0.021170,0.040836,HIGH_VISIBILITY_REVIEW,REVIEW_PERFORMANCE,Higher observed visibility; still requires review
4,5,client_2094c6eb080311d5,content_7f3e5c1ec603a7f1,181.0,12.0,0.066298,2.303867,1-3,0.0,-0.066298,0.035344,-0.030954,0.040271,MODEL_SIGNAL_REVIEW,REVIEW_TREND,Limited evidence; interpret cautiously
5,6,client_f623b01661d4bfe4,content_ed67a2a35929aaf7,786.0,39.0,0.049618,15.152672,11-20,0.0,-0.049618,0.025613,-0.024005,0.040018,STRIKING_DISTANCE_REVIEW,REVIEW_CONTENT,Moderate evidence; check context before acting
6,7,client_fef1a8f436438636,content_e38d62d8a90379ef,1397.0,71.0,0.050823,3.120258,4-10,0.0,-0.050823,0.028855,-0.021968,0.039777,HIGH_VISIBILITY_REVIEW,REVIEW_PERFORMANCE,Higher observed visibility; still requires review
7,8,client_fef1a8f436438636,content_4ca0dd5b44c124d9,1496.0,79.0,0.052807,3.881684,4-10,0.0,-0.052807,0.033996,-0.018811,0.034383,HIGH_VISIBILITY_REVIEW,REVIEW_PERFORMANCE,Higher observed visibility; still requires review
8,9,client_f623b01661d4bfe4,content_d359e26e27260345,107.0,5.0,0.046729,12.607477,11-20,0.0,-0.046729,0.018416,-0.028313,0.033141,STRIKING_DISTANCE_REVIEW,REVIEW_CONTENT,Limited evidence; interpret cautiously
9,10,client_f623b01661d4bfe4,content_cdbf00ce4999800f,123.0,7.0,0.056911,23.170732,21-50,0.0,-0.056911,0.030285,-0.026626,0.032086,MODEL_SIGNAL_REVIEW,REVIEW_TREND,Limited evidence; interpret cautiously


### How to interpret the queue

The output ranks eligible pages using the positive gap between observed CTR and the median CTR for the same position group, adjusted by search visibility. A smaller model-based component highlights pages whose estimated future CTR differs from current CTR.

The queue should be used to decide what to inspect first, not to automatically change a page. A reviewer should verify search intent, query mix, snippet relevance, seasonality, and the page’s actual content before approving an action.

The model signal is not a predicted uplift from editing a page. A predicted CTR above the observed CTR does not prove that an edit will improve performance, and a predicted CTR below the observed CTR does not prove that the page will decline.

This is a historical validation queue built from March features for held-out clients. The April target is used to train and validate the model, but April metrics are not used in the queue’s reason codes or priority score. The queue is therefore useful for a retrospective review exercise, but it should not be presented as a fully validated live deployment system.

## 2. Intended use and limits

### Intended use

The ranked action queue is a **human-review aid** for prioritizing content pages using observed Google Search Console performance. It highlights pages whose observed click-through rate (CTR) is below the typical CTR of pages in a similar average-position group, while accounting for search visibility.

The queue helps a reviewer decide **which pages to investigate first**. A recommendation such as `REVIEW_SNIPPET` means the page is worth checking for title and description clarity, search-intent alignment, and the context of its search results. It is not an instruction to edit the page automatically.

### Limits and assumptions

- **Association is not causation.** A below-peer CTR does not establish that the snippet caused the difference or that editing it will increase clicks.
- **Position groups are approximate.** Pages in the same position range may differ in query intent, brand recognition, device mix, SERP features, and competition.
- **Impressions affect evidence strength.** A CTR calculated from few impressions can change substantially after a small number of additional clicks. The minimum-impression threshold reduces, but does not eliminate, this uncertainty.
- **The score is a prioritization heuristic.** It is not a calibrated probability of success, an estimate of causal uplift, or a forecast of business impact.
- **Historical data may not represent current conditions.** Seasonality, search-result changes, algorithm updates, and recent content edits may make a historical recommendation less relevant.
- **Coverage is limited to the available signals.** The queue does not, by itself, establish whether a page is technically healthy, indexed, factually accurate, or aligned with business priorities.

### Operating boundary

Use the queue to select pages for investigation, not to make automatic content changes. A reviewer should check the page and its search context, record the decision, and choose to act, monitor, or reject the recommendation. If the available evidence is weak or important context is missing, monitoring is preferable to forcing an action.

In [ ]:
print("Pages in review queue:", len(queue))

if len(queue) > 0:
    print(
        "Impressions — minimum:",
        int(queue["gsc_impressions"].min()),
        "| median:",
        round(float(queue["gsc_impressions"].median()), 1),
        "| maximum:",
        int(queue["gsc_impressions"].max()),
    )

    print("\nReason-code distribution:")
    display(
        queue["reason_code"]
        .value_counts()
        .rename_axis("reason_code")
        .to_frame("page_count")
    )

    print("\nEvidence notes:")
    display(
        queue["evidence_note"]
        .value_counts()
        .rename_axis("evidence_note")
        .to_frame("page_count")
    )

    print(
        "\nReminder: these are prioritization signals, "
        "not evidence of causal uplift."
    )
else:
    print("The queue is empty; check candidate eligibility and scoring.")

Pages in review queue: 23496
Impressions — minimum: 100 | median: 704.0 | maximum: 617124

Reason-code distribution:


,page_count
reason_code,
MONITOR_NO_CLEAR_FLAG,10225
HIGH_VISIBILITY_REVIEW,7654
STRIKING_DISTANCE_REVIEW,4853
MODEL_SIGNAL_REVIEW,764



Evidence notes:


,page_count
evidence_note,
Higher observed visibility; still requires review,9373
Moderate evidence; check context before acting,8068
Limited evidence; interpret cautiously,6055



Reminder: these are prioritization signals, not evidence of causal uplift.


## 3. Human review + the no-go list

### Human review protocol

The ranked action queue is a decision-support tool. Every recommendation must be reviewed by a person before any content or SEO change is made.

For each queued page, the reviewer should:

1. **Verify the evidence:** Check impressions, clicks, observed CTR, average position, and the reason code. Confirm that the page meets the queue's eligibility criteria.
2. **Compare fairly:** Review CTR alongside pages with similar average positions. Where possible, inspect query mix, search intent, device differences, and SERP features before concluding that performance is unusually weak.
3. **Inspect the page:** Check whether the title and meta description accurately represent the content, satisfy the likely search intent, and communicate the page's value clearly.
4. **Check recent changes:** Look for recent content edits, migrations, indexing issues, seasonality, or other changes that could explain the observed performance. Respect the project's cooldown policy after optimization.
5. **Record a decision:** Choose one of the following outcomes:
   - **Act:** Evidence supports a specific, human-approved change.
   - **Monitor:** Evidence is insufficient, the page is too new, or the signal may be temporary.
   - **Reject:** The flag is explained by context or does not identify a meaningful issue.

Record the reviewer decision and rationale so that future queue reviews can distinguish useful flags from false alarms.

### No-go list

The system must **not**:

- Automatically rewrite, publish, or delete page content, titles, or meta descriptions.
- Treat below-peer CTR as proof that a snippet caused poor performance.
- Promise ranking gains, additional clicks, or business impact from a recommended change.
- Recommend changes based only on low CTR without considering average position, impressions, and search context.
- Treat model scores or priority scores as probabilities of success or estimates of causal uplift.
- Use future-period target values to select or prioritize current queue candidates.
- Override human judgment when data is sparse, contradictory, stale, or missing.
- Recommend factual, medical, legal, or other sensitive content changes without appropriate subject-matter review.

### Escalation rule

If the evidence is incomplete, the reason code is unclear, or the proposed change could affect factual accuracy or sensitive content, do not execute the recommendation. Escalate it to the appropriate content or subject-matter owner, or choose **Monitor** until sufficient evidence is available.

In [13]:
# Section 3: Prepare the human-review tracker

review_queue = queue.copy()

# Add fields for the human reviewer to complete
review_fields = {
    "review_status": "PENDING",
    "reviewer_decision": "",
    "reviewer_notes": "",
    "reviewer": "",
    "review_date": "",
}

for column, default_value in review_fields.items():
    if column not in review_queue.columns:
        review_queue[column] = default_value

print(f"Review records prepared: {len(review_queue):,}")
print("\nReview status:")
display(
    review_queue["review_status"]
    .value_counts(dropna=False)
    .rename_axis("review_status")
    .to_frame("page_count")
)

# Display the most useful columns first, when they exist
preferred_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "observed_ctr",
    "reason_code",
    "evidence_note",
    "review_status",
    "reviewer_decision",
    "reviewer_notes",
    "reviewer",
    "review_date",
]

display(
    review_queue[
        [col for col in preferred_columns if col in review_queue.columns]
    ].head(10)
)

Review records prepared: 23,496

Review status:


,page_count
review_status,
PENDING,23496


,rank,client_hash_id,content_hash_id,gsc_impressions,observed_ctr,reason_code,evidence_note,review_status,reviewer_decision,reviewer_notes,reviewer,review_date
0,1,client_f623b01661d4bfe4,content_0a124186f945b8cd,259.0,0.073359,STRIKING_DISTANCE_REVIEW,Limited evidence; interpret cautiously,PENDING,,,,
1,2,client_f623b01661d4bfe4,content_b96495c6b2911b12,160.0,0.075000,MODEL_SIGNAL_REVIEW,Limited evidence; interpret cautiously,PENDING,,,,
2,3,client_2094c6eb080311d5,content_04e3473c74438c83,140.0,0.078571,MODEL_SIGNAL_REVIEW,Limited evidence; interpret cautiously,PENDING,,,,
3,4,client_f623b01661d4bfe4,content_7b50820e76b1a006,2243.0,0.045921,HIGH_VISIBILITY_REVIEW,Higher observed visibility; still requires review,PENDING,,,,
4,5,client_2094c6eb080311d5,content_7f3e5c1ec603a7f1,181.0,0.066298,MODEL_SIGNAL_REVIEW,Limited evidence; interpret cautiously,PENDING,,,,
5,6,client_f623b01661d4bfe4,content_ed67a2a35929aaf7,786.0,0.049618,STRIKING_DISTANCE_REVIEW,Moderate evidence; check context before acting,PENDING,,,,
6,7,client_fef1a8f436438636,content_e38d62d8a90379ef,1397.0,0.050823,HIGH_VISIBILITY_REVIEW,Higher observed visibility; still requires review,PENDING,,,,
7,8,client_fef1a8f436438636,content_4ca0dd5b44c124d9,1496.0,0.052807,HIGH_VISIBILITY_REVIEW,Higher observed visibility; still requires review,PENDING,,,,
8,9,client_f623b01661d4bfe4,content_d359e26e27260345,107.0,0.046729,STRIKING_DISTANCE_REVIEW,Limited evidence; interpret cautiously,PENDING,,,,
9,10,client_f623b01661d4bfe4,content_cdbf00ce4999800f,123.0,0.056911,MODEL_SIGNAL_REVIEW,Limited evidence; interpret cautiously,PENDING,,,,


In [14]:
# Section 3: Validate review fields and permitted decisions

required_review_fields = [
    "review_status",
    "reviewer_decision",
    "reviewer_notes",
    "reviewer",
    "review_date",
]

missing = [
    col for col in required_review_fields
    if col not in review_queue.columns
]
assert not missing, f"Missing review fields: {missing}"

assert len(review_queue) == len(queue), (
    "The review tracker should preserve every queue row."
)

assert review_queue["review_status"].eq("PENDING").all(), (
    "New review records should start as PENDING."
)

allowed_decisions = {"", "ACT", "MONITOR", "REJECT"}
actual_decisions = set(
    review_queue["reviewer_decision"].fillna("").astype(str).str.upper()
)
invalid_decisions = actual_decisions - allowed_decisions

assert not invalid_decisions, (
    f"Unexpected reviewer decisions: {invalid_decisions}"
)

print("PASS: Review fields exist.")
print("PASS: No queue rows were lost or added.")
print("PASS: All new review records are pending.")
print("PASS: Reviewer decisions follow the allowed vocabulary.")
print("\nReminder: these checks validate the tracker structure,")
print("not the correctness of any individual recommendation.")

PASS: Review fields exist.
PASS: No queue rows were lost or added.
PASS: All new review records are pending.
PASS: Reviewer decisions follow the allowed vocabulary.

Reminder: these checks validate the tracker structure,
not the correctness of any individual recommendation.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.